# WP36 — Transformer-Based Policy Head (v0.4)
## MultiHeadSelfAttention · TransformerPolicyHead · SequenceBuffer

Demonstrates **WP36**: a lightweight pure-Python transformer that conditions synthesis action selection on the history of past (state, action) pairs.

Runtime: **~3 min** (no GPU, no PyTorch)

In [ ]:
import sys, os, importlib
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('Prometheus_v0_PoC'):
        os.system('git clone -b wp16-notebook-only https://github.com/pmcray/Prometheus_v0_PoC.git')
    os.chdir('Prometheus_v0_PoC')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
    importlib.invalidate_caches()
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    importlib.invalidate_caches()
import warnings; warnings.filterwarnings('ignore')
import time, random, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as mpatches
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (14, 7), 'font.size': 11})
SEED = 42; random.seed(SEED); np.random.seed(SEED)
import prometheus; print(f'Prometheus {prometheus.__version__} OK')

In [ ]:
from prometheus.wp36_transformer_policy import (
    MultiHeadSelfAttention, TransformerBlock, TransformerPolicyHead,
    TransformerTokeniser, SequenceBuffer, verify_wp36_exit_criteria,
)
from prometheus.wp17_crls_synthesis import SynthesisAction
N_ACTIONS = len(SynthesisAction)
D_STATE = 8; D_MODEL = 16
print(f'SynthesisActions: {[a.name for a in SynthesisAction]}')
print(f'n_actions={N_ACTIONS}  d_state={D_STATE}  d_model={D_MODEL}')

In [ ]:
# MultiHeadSelfAttention shape test
attn = MultiHeadSelfAttention(d_model=D_MODEL, n_heads=2, seed=0)
dummy_X = [[float(i*j+1)/10 for j in range(D_MODEL)] for i in range(5)]
out = attn.forward(dummy_X)
print(f'MHSA input shape:  ({len(dummy_X)}, {len(dummy_X[0])})')
print(f'MHSA output shape: ({len(out)}, {len(out[0])})')
print(f'Output non-zero:   {any(abs(v)>0.001 for row in out for v in row)}')

In [ ]:
# TransformerPolicyHead forward
tf = TransformerPolicyHead(d_state=D_STATE, n_actions=N_ACTIONS, d_model=D_MODEL, n_layers=2, n_heads=2)
history = [([float(i%4)/4]*D_STATE, i % N_ACTIONS) for i in range(6)]
logits = tf.forward(history)
probs  = tf.softmax_policy(history)
greedy = tf.greedy_action(history)
print(f'Logits  ({len(logits)} dims): {[round(x,3) for x in logits]}')
print(f'Probs   ({len(probs)} dims): {[round(x,3) for x in probs]}')
print(f'Greedy action: {greedy} = {list(SynthesisAction)[greedy].name}')
print(f'Max-min prob gap: {max(probs)-min(probs):.4f} (>0 confirms non-uniform)')

In [ ]:
# SequenceBuffer rolling window
buf = SequenceBuffer(max_seq_len=8, d_state=D_STATE)
print('SequenceBuffer (max_seq_len=8):')
for step in range(12):
    sv = [float(step)/10]*D_STATE
    buf.push(sv, step % N_ACTIONS)
    if step in (3, 7, 11):
        print(f'  After {step+1} pushes: len={len(buf)} (capped at 8)')

In [ ]:
# Simulate 15 generations with transformer policy
import time
gen_records = []
buf2 = SequenceBuffer(max_seq_len=16, d_state=D_STATE)
tf2  = TransformerPolicyHead(d_state=D_STATE, n_actions=N_ACTIONS, d_model=D_MODEL, n_layers=2)

for gen in range(15):
    # Fake state: accuracy improves over time
    acc = 0.5 + gen * 0.02 + random.gauss(0, 0.02)
    sv  = [acc, 0.20, 0.15, 1.5, 0.30, 0.10, gen/100, len(buf2)/16]
    if len(buf2) >= 4:
        probs   = tf2.softmax_policy(buf2.get())
        tf_act  = probs.index(max(probs))
        source  = 'transformer'
        top_p   = max(probs)
    else:
        tf_act  = random.randint(0, N_ACTIONS-1)
        source  = 'random_warmup'
        top_p   = 1/N_ACTIONS
    buf2.push(sv, tf_act)
    gen_records.append({'gen':gen,'source':source,'action':list(SynthesisAction)[tf_act].name,'top_p':top_p,'acc':acc})
    print(f'  Gen {gen:02d}  [{source:<15}]  action={list(SynthesisAction)[tf_act].name:<15}  p={top_p:.3f}  acc={acc:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
gens = [r['gen'] for r in gen_records]

ax = axes[0]
ax.plot(gens, [r['acc'] for r in gen_records], 'o-', color='#4CAF50', label='Accuracy')
ax.set_xlabel('Generation'); ax.set_ylabel('Accuracy')
ax.set_title('Agent Accuracy over Generations', fontweight='bold'); ax.legend()

ax2 = axes[1]
ax2.bar(gens, [r['top_p'] for r in gen_records],
        color=['#2196F3' if r['source']=='transformer' else '#9E9E9E' for r in gen_records],
        edgecolor='black', alpha=0.85)
ax2.set_xlabel('Generation'); ax2.set_ylabel('Top Action Prob')
ax2.set_title('Transformer Action Confidence per Generation', fontweight='bold')
ax2.legend(handles=[mpatches.Patch(color='#2196F3',label='Transformer'),
                    mpatches.Patch(color='#9E9E9E',label='Warmup')])
fig.suptitle('WP36: Transformer Policy Head', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('wp36_transformer_policy.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved wp36_transformer_policy.png')

In [ ]:
from prometheus.wp36_transformer_policy import TransformerRecord, verify_wp36_exit_criteria
dummy_recs = [TransformerRecord(gen,
    'transformer' if r['source']=='transformer' else 'crls_upstream',
    list(SynthesisAction)[r['gen']%N_ACTIONS], list(SynthesisAction)[0],
    min(gen+1,8), r['top_p'])
    for gen, r in enumerate(gen_records)]
criteria = verify_wp36_exit_criteria(dummy_recs, buf2, tf2)
print('WP36 Exit Criteria Verification'); print('='*60)
for c, ok in criteria.items():
    print(f'  {"PASS" if ok else "FAIL"}  {c}')
if all(criteria.values()):
    print('\nAll WP36 exit criteria satisfied.')

---
## Conclusions

**WP36** adds sequential context modelling to the synthesis stack:
- Pure-Python multi-head self-attention (no PyTorch dependency)
- `SequenceBuffer` provides rolling (state, action) history
- `TransformerPolicyHead` conditions action selection on history

### Foundation for WP40
WP40 replaces the rule-based `InvariantGuard` (WP27) with a learned safety classifier — trained on the same (state, action, outcome) triples that WP36 sequences.